# Droplet Analysis Pipeline
1. Install packages & imports
2. Set filename and pixel size
3. Run detection + classification pipeline
4. Interactive figure
5. Edit widgets (update class / merge / add / delete)
6. Statistical analysis

In [5]:
# Uncomment if running in Google Colab:
# from google.colab import output
# output.enable_custom_widget_manager()

!pip install czifile scikit-learn -q

import numpy as np
import pandas as pd
import czifile
from scipy import ndimage as ndi
from scipy.stats import norm as sp_norm
from scipy.optimize import brentq
from skimage import filters, morphology
from skimage.feature import peak_local_max
from sklearn.mixture import GaussianMixture
import plotly.graph_objects as go
import ipywidgets as widgets
import matplotlib.pyplot as plt
from IPython.display import display, clear_output


[notice] A new release of pip is available: 23.2.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


# Set file and imaging parameters

In [7]:
filename     = "your_image.czi"
pixel_size   = 0.312   # µm/pixel  (0.312 for 20x, 0.156 for 40x)

# Detection & Classification Pipeline

In [8]:
# ═══════════════════════════════════════════════════════════════
# IMPROVED PIPELINE
# ═══════════════════════════════════════════════════════════════

# ── 1. Load & normalise ──────────────────────────────────────
def load_image(filename):
    img = np.squeeze(czifile.imread(filename)).astype(float)
    assert img.ndim == 3, f"Expected (C,H,W), got {img.shape}"
    return img

def normalize_image(img):
    out = np.zeros_like(img)
    for c in range(img.shape[0]):
        lo, hi = np.percentile(img[c], [1, 99])
        out[c] = np.clip((img[c] - lo) / (hi - lo + 1e-9), 0, 1)
    return out

# ── 2. Detect droplets — union of per-channel masks ──────────
def detect_droplets(norm, pixel_size_um,
                    sigma_override=None, minsize_override=None, mind_override=None):
    """Returns centers (N,2), radii (N,), distance map (H,W).
    Uses a union of per-channel Otsu masks so droplets visible
    in only one dim channel are still detected.
    sigma_override, minsize_override, mind_override: slider values that
    replace the pixel-size-derived defaults when provided.
    """
    n_ch, H, W = norm.shape
    sigma    = sigma_override    if sigma_override    is not None else max(1.0, 2.0 * 0.312 / pixel_size_um)
    min_size = minsize_override  if minsize_override  is not None else max(20,  int(20 * (0.312 / pixel_size_um) ** 2))
    min_d    = mind_override     if mind_override     is not None else max(5,   int(5.0 / pixel_size_um))

    ch_masks = []
    for c in range(n_ch):
        sm = filters.gaussian(norm[c], sigma=sigma)
        t  = filters.threshold_otsu(sm)
        m  = morphology.remove_small_objects(sm > t, min_size=min_size)
        ch_masks.append(m)
    union_mask = np.logical_or.reduce(ch_masks)

    dist   = ndi.distance_transform_edt(union_mask)
    coords = peak_local_max(dist, min_distance=min_d, labels=union_mask)

    # Keep highest-distance peak when two centres are within min_d
    order  = np.argsort(-dist[coords[:, 0], coords[:, 1]])
    coords = coords[order]
    kept = []
    for r, c in coords:
        if all(np.hypot(r - r0, c - c0) > min_d for r0, c0 in kept):
            kept.append((r, c))

    if not kept:
        return np.empty((0, 2), int), np.empty(0), dist

    centers = np.array(kept, dtype=int)
    radii   = dist[centers[:, 0], centers[:, 1]]
    return centers, radii, dist

# ── 3. Per-droplet intensity ──────────────────────────────────
def measure_droplet_intensities(img, centers, radii):
    """Mean raw intensity inside each droplet's circular ROI."""
    n_ch, H, W = img.shape
    yy, xx = np.mgrid[0:H, 0:W]
    intensities = np.zeros((len(centers), n_ch))
    for i, ((r, c), rad) in enumerate(zip(centers, radii)):
        mask = (yy - r) ** 2 + (xx - c) ** 2 <= rad ** 2
        intensities[i] = [img[ch][mask].mean() for ch in range(n_ch)]
    return intensities

# ── 4. Background estimation — erosion-based ─────────────────
def estimate_background(img, centers, radii, pixel_size_um, excl_factor=2.0):
    """Background pixels are at least 2× droplet-radius away from
    every centre, then further eroded to avoid halo contamination."""
    n_ch, H, W = img.shape
    yy, xx = np.mgrid[0:H, 0:W]
    excl = np.zeros((H, W), bool)
    for (r, c), rad in zip(centers, radii):
        excl |= (yy - r) ** 2 + (xx - c) ** 2 <= (rad * excl_factor) ** 2
    erode_px = max(3, int(1.0 / pixel_size_um))
    excl     = morphology.binary_dilation(excl, morphology.disk(erode_px))
    bg = ~excl
    if bg.sum() < 500:               # fallback if image is very dense
        excl2 = np.zeros((H, W), bool)
        for (r, c), rad in zip(centers, radii):
            excl2 |= (yy - r) ** 2 + (xx - c) ** 2 <= (rad * 1.5) ** 2
        bg = ~excl2
    bg_means = np.array([img[ch][bg].mean() for ch in range(n_ch)])
    bg_stds  = np.array([img[ch][bg].std()  for ch in range(n_ch)])
    return bg_means, bg_stds

# ── 5. GMM classification per channel ────────────────────────
def classify_channels(intensities, bg_means, bg_stds, reliability_sigma=1.0):
    """Fit a 2-component GMM on bg-subtracted per-droplet intensities
    for each channel. Threshold = intersection of the two Gaussians.
    Channels whose positive-component mean < 1×bg_std are flagged
    unreliable (binary value set to -1).

    Returns binary (N,n_ch), thresholds (n_ch,), reliable (n_ch,).
    """
    n_ch   = intensities.shape[1]
    bg_sub = intensities - bg_means
    binary     = np.zeros_like(bg_sub, dtype=int)
    thresholds = np.zeros(n_ch)
    reliable   = np.ones(n_ch, dtype=bool)

    for c in range(n_ch):
        vals = bg_sub[:, c]
        if vals.max() <= vals.min():
            reliable[c] = False
            continue

        gmm = GaussianMixture(n_components=2, random_state=0,
                              n_init=10, max_iter=200).fit(vals.reshape(-1, 1))
        means = gmm.means_.flatten()
        order = np.argsort(means)
        m0, m1 = means[order]
        w0, w1 = gmm.weights_[order]
        s0 = np.sqrt(gmm.covariances_[order[0], 0, 0])
        s1 = np.sqrt(gmm.covariances_[order[1], 0, 0])

        def f(x):
            return w0 * sp_norm.pdf(x, m0, s0) - w1 * sp_norm.pdf(x, m1, s1)
        try:
            thresh = brentq(f, m0, m1)
        except Exception:
            thresh = (m0 + m1) / 2.0

        thresholds[c] = thresh
        binary[:, c]  = (vals > thresh).astype(int)

        if m1 < reliability_sigma * bg_stds[c]:  # positive mean barely above noise
            reliable[c] = False

    return binary, thresholds, reliable

# ── 6. Bright-red detection ───────────────────────────────────
def detect_bright_red(intensities, bg_means, bg_stds, red_ch=0,
                      bright_factor=2.0, bright_pct=98.0):
    """3-component GMM on the red channel to separate:
    negative / dim-red / bright-red.
    A droplet is labelled bright-red only when the highest-mean
    component is well-separated (> bright_factor× the dim-red mean).
    """
    vals = (intensities[:, red_ch] - bg_means[red_ch]).reshape(-1, 1)
    gmm3    = GaussianMixture(n_components=3, random_state=0,
                              n_init=10, max_iter=200).fit(vals)
    means3  = gmm3.means_.flatten()
    order3  = np.argsort(means3)
    labels3 = gmm3.predict(vals)

    bright_comp = order3[2]
    dim_mean    = means3[order3[1]]
    bright_mean = means3[order3[2]]

    if bright_mean > 0 and dim_mean > 0 and bright_mean > bright_factor * dim_mean:
        bright_red = (labels3 == bright_comp)
    else:
        # Fallback: top 2% AND above 2× dim-red mean
        threshold  = np.percentile(vals, bright_pct)
        bright_red = (vals.flatten() > threshold) & (vals.flatten() > bright_factor * dim_mean)

    return bright_red

# ── 7. Assemble dataframe ─────────────────────────────────────
def build_dataframe(centers, radii, intensities, binary,
                    bright_red, reliable, pixel_size_um):
    code_ints = (binary[:, 0] * 8 + binary[:, 1] * 4 +
                 binary[:, 2] * 2 + binary[:, 3] * 1).astype(int)
    code_ints[bright_red] = 0           # bright red → class 0

    rows = []
    for i, ((r, c), rad) in enumerate(zip(centers, radii)):
        row = dict(
            label      = i + 1,
            centroid_y = int(r),
            centroid_x = int(c),
            radius     = float(rad),
            radius_um  = float(rad * pixel_size_um),
            volume_um3 = float((4/3) * np.pi * (rad * pixel_size_um) ** 3),
            bright_red = bool(bright_red[i]),
            code_int   = int(code_ints[i]),
            intensity_ch1 = float(intensities[i, 0]),
            intensity_ch2 = float(intensities[i, 1]),
            intensity_ch3 = float(intensities[i, 2]),
            intensity_ch4 = float(intensities[i, 3]),
        )
        for ci, ch in enumerate(['binary_ch1','binary_ch2','binary_ch3','binary_ch4']):
            row[ch] = int(binary[i, ci]) if reliable[ci] else -1
        rows.append(row)

    return pd.DataFrame(rows)

# ── 8. RGB composite ──────────────────────────────────────────
def make_rgb(norm):
    channel_colors = [[1,0,0],[1,1,0],[0,1,1],[0,0,1]]
    n_ch, H, W = norm.shape
    rgb = np.zeros((H, W, 3))
    for c in range(min(n_ch, 4)):
        for j in range(3):
            rgb[:, :, j] += norm[c] * channel_colors[c][j]
    return (np.clip(rgb, 0, 1) * 255).astype(np.uint8)

# ── 9. Class colours (0 = bright red) ────────────────────────
HEX_COLORS = [
    "#ff2222","#1d1b7d","#086564","#15a2ff","#b8b910","#6a6a69",
    "#81f66c","#95d9ab","#530409","#720351","#7db8c0","#798ced",
    "#ffd425","#faa34b","#de963e","#f7e5b6"
]

# ═══════════════════════════════════════════════════════════════
# RUN PIPELINE
# ═══════════════════════════════════════════════════════════════
image        = load_image(filename)
norm         = normalize_image(image)
H, W         = image.shape[1], image.shape[2]

centers, radii, dist_map = detect_droplets(norm, pixel_size)
intensities              = measure_droplet_intensities(image, centers, radii)
bg_means, bg_stds        = estimate_background(image, centers, radii, pixel_size)
binary, thresholds, reliable = classify_channels(intensities, bg_means, bg_stds)
bright_red               = detect_bright_red(intensities, bg_means, bg_stds)
df                       = build_dataframe(centers, radii, intensities,
                                           binary, bright_red, reliable, pixel_size)

print(f"Image shape   : {image.shape}")
print(f"Droplets found: {len(df)}")
print(f"Classes found : {sorted(df.code_int.unique())} ({df.code_int.nunique()}/16)")
print(f"Reliable chs  : {reliable}  (False = SNR too low)")
print(f"Bright red    : {df.bright_red.sum()}")
print()
display(df.head())

# Create RGB image for figure base
rgb_image_uint8 = make_rgb(norm)

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\duyan\\Documents\\Postdoc\\2026_Summer\\16_NS_Experiment\\Data_analysis\\your_image.czi'

# Interactive UI
Run the cell below to launch the full interface.

In [9]:
# ═══════════════════════════════════════════════════════════════
# INTERACTIVE UI
# ═══════════════════════════════════════════════════════════════
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display

HEX = HEX_COLORS   # alias

# ── Shared mutable state ─────────────────────────────────────
S = {
    "df": df, "norm": norm, "img": image,
    "bg_means": bg_means, "bg_stds": bg_stds, "reliable": reliable,
}

# ── Helper: build uint8 RGB composite ────────────────────────
def _make_rgb(norm_arr):
    return make_rgb(norm_arr)

# ── Figure widget ─────────────────────────────────────────────
_, H0, W0 = norm.shape
fig = go.FigureWidget(layout=go.Layout(
    width=880, height=880,
    margin=dict(l=0, r=0, t=30, b=0),
    xaxis=dict(range=[0, W0], constrain="domain", showgrid=False, zeroline=False),
    yaxis=dict(range=[0, H0], autorange="reversed", scaleanchor="x", showgrid=False, zeroline=False),
    title_text="Run pipeline to display",
    dragmode="zoom",
))

# ── draw_droplets — FigureWidget-safe implementation ─────────
#    FigureWidget forbids assigning fig.data to a list of NEW
#    trace objects. Fix:
#      1. update image trace in-place (.z = ...)
#      2. subset to just the image:  fig.data = (fig.data[0],)
#         (valid because it is a subset of existing traces)
#      3. batch-add fresh scatter traces with fig.add_traces()
def draw_droplets():
    df_cur, norm_cur = S["df"], S["norm"]
    if df_cur is None or norm_cur is None:
        return
    _, H, W = norm_cur.shape
    new_rgb = _make_rgb(norm_cur)

    if len(fig.data) == 0:
        fig.add_trace(go.Image(z=new_rgb))
    else:
        fig.data[0].z = new_rgb        # update image data in-place
        fig.data = (fig.data[0],)      # drop scatter traces (valid subset)

    shapes   = []
    scatters = []
    for _, row in df_cur.iterrows():
        col = HEX[int(row.code_int) % 16]
        shapes.append(dict(
            type="circle",
            x0=row.centroid_x - row.radius, y0=row.centroid_y - row.radius,
            x1=row.centroid_x + row.radius, y1=row.centroid_y + row.radius,
            line=dict(color=col, width=2),
        ))
        scatters.append(go.Scatter(
            x=[row.centroid_x], y=[row.centroid_y],
            text=[str(int(row.code_int))], mode="text",
            textfont=dict(color="white", size=10),
            hovertext=(f"ID:{int(row.label)}  class:{int(row.code_int)}<br>"
                       f"r={row.radius:.1f}px  vol={row.volume_um3:.2f} µm³<br>"
                       f"bright_red:{row.bright_red}"),
            hoverinfo="text",
        ))

    if scatters:
        fig.add_traces(scatters)

    rel = S.get("reliable")
    unrel = (" | ⚠ " + ",".join(f"ch{i+1}" for i, r in enumerate(rel) if not r)
             + " unreliable") if rel is not None and not all(rel) else ""
    fig.update_layout(
        shapes=shapes,
        xaxis_range=[0, W],
        yaxis_range=[0, H],
        title_text=(f"{len(df_cur)} droplets | "
                    f"{df_cur.code_int.nunique()}/16 classes" + unrel),
    )
    for t in fig.data[1:]:
        t.on_click(_on_click)

# ── Click handler ─────────────────────────────────────────────
def _on_click(trace, points, selector):
    if not points.point_inds:
        return
    df_cur = S["df"]
    if df_cur is None:
        return
    # Scatter traces start at index 1; each droplet is one trace
    tidx = list(fig.data).index(trace) - 1
    if 0 <= tidx < len(df_cur):
        row = df_cur.iloc[tidx]
        sel_id_box.value = int(row.label)
        class_dd.value   = int(row.code_int)

# ── Detection sliders ─────────────────────────────────────────
sigma_sl   = widgets.FloatSlider(value=1.0, min=0.3, max=5.0, step=0.1,
                                  description="σ (blur)", style={"description_width":"120px"}, layout=widgets.Layout(width="420px"))
minsize_sl = widgets.IntSlider(  value=20,  min=5,   max=200,  step=5,
                                  description="min px²", style={"description_width":"120px"}, layout=widgets.Layout(width="420px"))
mind_sl    = widgets.IntSlider(  value=5,   min=2,   max=50,   step=1,
                                  description="min dist px", style={"description_width":"120px"}, layout=widgets.Layout(width="420px"))
excl_sl    = widgets.FloatSlider(value=2.0, min=1.0, max=5.0,  step=0.1,
                                  description="BG excl ×r", style={"description_width":"120px"}, layout=widgets.Layout(width="420px"))
relth_sl   = widgets.FloatSlider(value=1.0, min=0.0, max=5.0,  step=0.1,
                                  description="Reliability σ", style={"description_width":"120px"}, layout=widgets.Layout(width="420px"))
brcomp_sl  = widgets.FloatSlider(value=2.0, min=1.1, max=5.0,  step=0.1,
                                  description="Bright-red ×", style={"description_width":"120px"}, layout=widgets.Layout(width="420px"))
brpct_sl   = widgets.FloatSlider(value=98,  min=80,  max=99.9, step=0.5,
                                  description="Bright-red pct", style={"description_width":"120px"}, layout=widgets.Layout(width="420px"))

run_btn   = widgets.Button(description="▶ Run pipeline",    button_style="primary")
reclf_btn = widgets.Button(description="↺ Re-classify only",button_style="info")
save_btn  = widgets.Button(description="💾 Save CSV",        button_style="success")
status_lbl= widgets.Label(value="Ready.")

# ── Run / re-classify callbacks ───────────────────────────────
def run_pipeline_cb(_):
    status_lbl.value = "Running pipeline…"
    try:
        img = S["img"]
        if img is None:
            img = load_image(filename)
            S["img"] = img
        n = normalize_image(img)
        S["norm"] = n
        c, r, _ = detect_droplets(n, pixel_size,
                                   sigma_override=sigma_sl.value,
                                   minsize_override=minsize_sl.value,
                                   mind_override=mind_sl.value)
        inten = measure_droplet_intensities(img, c, r)
        bgm, bgs = estimate_background(img, c, r, pixel_size,
                                        excl_factor=excl_sl.value)
        S["bg_means"], S["bg_stds"] = bgm, bgs
        bn, _, rel = classify_channels(inten, bgm, bgs,
                                        reliability_sigma=relth_sl.value)
        S["reliable"] = rel
        br = detect_bright_red(inten, bgm, bgs,
                                bright_factor=brcomp_sl.value,
                                bright_pct=brpct_sl.value)
        S["df"] = build_dataframe(c, r, inten, bn, br, rel, pixel_size)
        draw_droplets()
        status_lbl.value = f"Done — {len(S['df'])} droplets, {S['df'].code_int.nunique()}/16 classes."
    except Exception as e:
        status_lbl.value = f"Error: {e}"
        raise

def reclassify_cb(_):
    status_lbl.value = "Re-classifying…"
    try:
        df_old = S["df"]
        if df_old is None:
            status_lbl.value = "No data — run pipeline first."
            return
        img = S["img"]
        inten = np.column_stack([df_old[f"intensity_ch{i+1}"] for i in range(4)])
        bgm, bgs = S["bg_means"], S["bg_stds"]
        bn, _, rel = classify_channels(inten, bgm, bgs,
                                        reliability_sigma=relth_sl.value)
        S["reliable"] = rel
        br = detect_bright_red(inten, bgm, bgs,
                                bright_factor=brcomp_sl.value,
                                bright_pct=brpct_sl.value)
        centers = df_old[["centroid_y","centroid_x"]].values
        radii   = df_old["radius"].values
        S["df"] = build_dataframe(centers, radii, inten, bn, br, rel, pixel_size)
        draw_droplets()
        status_lbl.value = f"Re-classified — {S['df'].code_int.nunique()}/16 classes."
    except Exception as e:
        status_lbl.value = f"Error: {e}"
        raise

def save_cb(_):
    out = filename.replace(".czi","_analysis.csv")
    S["df"].to_csv(out, index=False)
    status_lbl.value = f"Saved → {out}"

run_btn.on_click(run_pipeline_cb)
reclf_btn.on_click(reclassify_cb)
save_btn.on_click(save_cb)

# ─────────────────────────────────────────────────────────────────────────────
# EDIT CALLBACKS
# ─────────────────────────────────────────────────────────────────────────────
def _update_class(_):
    df = S["df"]
    if df is None: return
    did, cls = sel_id_box.value, class_dd.value
    if did not in df.label.values: return
    idx = df.index[df.label == did][0]
    S["df"].at[idx, "code_int"] = cls
    S["df"].at[idx, "bright_red"] = (cls == 0)
    draw_droplets()
    status_lbl.value = f"ID {did} → class {cls}"

def _merge_cb(_):
    df = S["df"]
    if df is None: return
    try:
        ids = [int(x.strip()) for x in merge_box.value.split(",") if x.strip()]
    except ValueError:
        status_lbl.value = "Enter comma-separated IDs."
        return
    if len(ids) < 2: return
    rows = df[df.label.isin(ids)]
    if rows.empty: return
    cy = rows.centroid_y.mean(); cx = rows.centroid_x.mean()
    r  = rows.radius.mean()
    keep = rows.iloc[0].copy()
    keep["centroid_y"] = cy; keep["centroid_x"] = cx; keep["radius"] = r
    keep["radius_um"]  = r * pixel_size
    keep["volume_um3"] = (4/3)*3.14159*(r*pixel_size)**3
    S["df"] = df[~df.label.isin(ids[1:])].copy()
    ki = S["df"].index[S["df"].label == ids[0]][0]
    for col in keep.index:
        S["df"].at[ki, col] = keep[col]
    draw_droplets()
    status_lbl.value = f"Merged {ids} → ID {ids[0]}"

def _add_cb(_):
    df = S["df"]
    if df is None: return
    try:
        y, x, r, cls = int(ay.value), int(ax.value), float(ar.value), int(ac.value)
    except ValueError:
        status_lbl.value = "Enter valid y, x, radius, class."
        return
    new_id = int(df.label.max()) + 1 if len(df) else 1
    img = S["img"]
    n_ch = img.shape[0]
    H2, W2 = img.shape[1], img.shape[2]
    yy, xx = np.mgrid[0:H2, 0:W2]
    mask = (yy-y)**2 + (xx-x)**2 <= r**2
    inten = np.array([img[c][mask].mean() for c in range(n_ch)])
    bgm = S["bg_means"]
    new_row = dict(label=new_id, centroid_y=y, centroid_x=x,
                   radius=r, radius_um=r*pixel_size,
                   volume_um3=(4/3)*3.14159*(r*pixel_size)**3,
                   bright_red=(cls==0), code_int=cls,
                   intensity_ch1=float(inten[0]), intensity_ch2=float(inten[1]),
                   intensity_ch3=float(inten[2]), intensity_ch4=float(inten[3]),
                   binary_ch1=0, binary_ch2=0, binary_ch3=0, binary_ch4=0)
    S["df"] = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)
    draw_droplets()
    status_lbl.value = f"Added ID {new_id} class {cls} at ({x},{y})"

def _delete_cb(_):
    df = S["df"]
    if df is None: return
    did = del_id_box.value
    if did not in df.label.values:
        status_lbl.value = f"ID {did} not found."
        return
    S["df"] = df[df.label != did].copy()
    draw_droplets()
    status_lbl.value = f"Deleted ID {did}"

# ── Edit controls ─────────────────────────────────────────────
sel_id_box = widgets.BoundedIntText(value=1, min=1, max=9999, description="Select ID:",
                                     style={"description_width":"80px"}, layout=widgets.Layout(width="200px"))
class_dd   = widgets.BoundedIntText(value=0, min=0, max=15, description="Class (0-15):",
                                     style={"description_width":"90px"}, layout=widgets.Layout(width="200px"))
upd_btn    = widgets.Button(description="Update class", button_style="warning", layout=widgets.Layout(width="140px"))
upd_btn.on_click(_update_class)

merge_box = widgets.Text(placeholder="e.g. 3,7", description="Merge IDs:",
                          style={"description_width":"80px"}, layout=widgets.Layout(width="260px"))
merge_btn = widgets.Button(description="Merge", button_style="warning", layout=widgets.Layout(width="80px"))
merge_btn.on_click(_merge_cb)

ay = widgets.IntText(value=0, description="y:", style={"description_width":"20px"}, layout=widgets.Layout(width="120px"))
ax = widgets.IntText(value=0, description="x:", style={"description_width":"20px"}, layout=widgets.Layout(width="120px"))
ar = widgets.FloatText(value=10, description="r:", style={"description_width":"20px"}, layout=widgets.Layout(width="120px"))
ac = widgets.BoundedIntText(value=1, min=0, max=15, description="cls:", style={"description_width":"25px"}, layout=widgets.Layout(width="100px"))
add_btn = widgets.Button(description="Add", button_style="success", layout=widgets.Layout(width="70px"))
add_btn.on_click(_add_cb)

del_id_box = widgets.BoundedIntText(value=1, min=1, max=9999, description="Delete ID:",
                                     style={"description_width":"80px"}, layout=widgets.Layout(width="200px"))
del_btn    = widgets.Button(description="Delete", button_style="danger", layout=widgets.Layout(width="80px"))
del_btn.on_click(_delete_cb)

# ── Layout ────────────────────────────────────────────────────
ctrl_panel = widgets.VBox([
    widgets.HTML("<b>Detection parameters</b>"),
    sigma_sl, minsize_sl, mind_sl, excl_sl, relth_sl, brcomp_sl, brpct_sl,
    widgets.HBox([run_btn, reclf_btn, save_btn]),
    widgets.HTML("<hr><b>Edit droplets</b>"),
    widgets.HBox([sel_id_box, class_dd, upd_btn]),
    widgets.HBox([merge_box, merge_btn]),
    widgets.HTML("<i>Add new:</i>"),
    widgets.HBox([ay, ax, ar, ac, add_btn]),
    widgets.HBox([del_id_box, del_btn]),
    widgets.HTML("<hr>"),
    status_lbl,
], layout=widgets.Layout(width="460px", padding="8px"))

ui = widgets.HBox([fig, ctrl_panel])
display(ui)

# Draw with initial pipeline results
draw_droplets()


    'data': [],
    'layout': {'dragmode': 'zoom',
               'height': 880,…

# Statistics
Run the cell below for size/count distributions.

In [ ]:
def analyze_droplets(df, pixel_size_um):
    """Summary statistics and plots for the classified droplets."""
    import matplotlib.pyplot as plt

    n_classes = df.code_int.nunique()
    print(f"Total droplets : {len(df)}")
    print(f"Classes found  : {n_classes}/16")
    print(f"Mean radius    : {df.radius_um.mean():.2f} ± {df.radius_um.std():.2f} µm")
    print()

    counts = df.groupby("code_int").size().reindex(range(16), fill_value=0)
    print("Counts per class:")
    print(counts.to_string())
    print()

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    # 1. Class counts
    ax = axes[0]
    colors = [HEX_COLORS[i] for i in range(16)]
    ax.bar(counts.index, counts.values, color=colors, edgecolor="k", linewidth=0.5)
    ax.set_xlabel("Class (code_int)")
    ax.set_ylabel("Count")
    ax.set_title("Droplets per class")
    ax.set_xticks(range(16))

    # 2. Radius distribution
    ax = axes[1]
    ax.hist(df.radius_um, bins=30, color="steelblue", edgecolor="k", linewidth=0.5)
    ax.set_xlabel("Radius (µm)")
    ax.set_ylabel("Count")
    ax.set_title("Radius distribution")

    # 3. Volume distribution
    ax = axes[2]
    ax.hist(df.volume_um3, bins=30, color="darkorange", edgecolor="k", linewidth=0.5)
    ax.set_xlabel("Volume (µm³)")
    ax.set_ylabel("Count")
    ax.set_title("Volume distribution")

    plt.tight_layout()
    plt.show()

    return counts

counts = analyze_droplets(S["df"], pixel_size)
